In [1]:
import os
import pandas as pd
import sys
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.widgets import Button

sys.path.insert(1, '../')
from functions import *

in_dir = '../example_station/input'
station_id = '026017'
qc_column = 1
qc_flags = '238'
round_decimals =1

path_in = os.path.join(in_dir, f"{station_id}.csv")
if not os.path.isfile(path_in):
    raise FileNotFoundError(f"Missing station file: {path_in}")

df = pd.read_csv(path_in, parse_dates=["datetime"], index_col="datetime")
for col in ("value", "flag"):
    if col not in df.columns:
        raise KeyError(f"Required column '{col}' not found in {path_in}")

In [14]:
if isinstance(station_id, str):
    station_list = [x.strip() for x in station_id.split(",") if x.strip()]
else:
    station_list = []
    for item in station_id:
        station_list.extend([x.strip() for x in str(item).split(",") if x.strip()])

all_results = []

for sid in station_list:
    path = os.path.join(in_dir, f"{sid}.csv")
    if not os.path.isfile(path):
        print(f"⚠️ Missing file for station {sid}: {path}")
        continue

    df = pd.read_csv(path, parse_dates=["datetime"], index_col="datetime")

    # Ensure required columns exist
    for col in ("value", "flag"):
        if col not in df.columns:
            print(f"⚠️ Column '{col}' not found in {path}; skipping station {sid}.")
            break
    else:
        # Remove rows where value == 'NA' (string)
        df = remove_empty(df)

        # Normalize flag to zero-padded 3-character strings (e.g., 0 -> '000', 60 -> '060')
        flag_str = pd.to_numeric(df["flag"], errors="coerce").fillna(0).astype(int).astype(str).str.zfill(3)

        # Infer timestep from the full series index
        expected_delta = infer_time_step(df.index)

        # Build QC mask on normalized flags
        mask = build_qc_mask(flag_str, qc_column=qc_column, qc_flags=qc_flags)
        times = df.index[mask].sort_values()

        if times.empty:
            print(
                f"ℹ️ No QC flags matching column {qc_column} with digits '{qc_flags}' "
                f"for station {sid}."
            )
            continue

        # Group into consecutive runs
        runs = group_consecutive_times(times, expected_delta)
        print(
            f"👉 Station {sid}: {len(runs)} chunk(s) to review "
            f"(qc_column={qc_column}, qc_flags='{qc_flags}', step={expected_delta})."
        )

        # Interactive labeling for each run
        for start, end in runs:
            # ... build fig/ax as before ...
            plot_out = widgets.Output()
            btn_out = widgets.Output()

            btn_true  = widgets.Button(description="True",  button_style="success")
            btn_false = widgets.Button(description="False", button_style="danger")

            with plot_out:
                fig, ax = plt.subplots(figsize=(10, 4))
                ax.plot(window.index, window.values, lw=1)
                ax.axvspan(start, end, color="red", alpha=0.3, label="Chunk")
                ax.set_title(f"{sid} — {start:%Y-%m-%d %H:%M} to {end:%Y-%m-%d %H:%M}")
                ax.set_xlabel("Datetime")
                ax.set_ylabel("Value")
                ax.legend(loc="upper left")
                plt.tight_layout()
                plt.show()

            def on_true(b, st=sid, s=start, e=end):
                all_results.append({"station": st, "start": s, "end": e, "real": True,})
                with btn_out: print(f"✅ Marked True: {s} – {e}")
                plot_out.clear_output()
                btn_out.clear_output()

            def on_false(b, st=sid, s=start, e=end):
                all_results.append({"station": st, "start": s, "end": e, "real": False,})
                with btn_out: print(f"❌ Marked False: {s} – {e}")
                plot_out.clear_output()
                btn_out.clear_output()

            btn_true.on_click(on_true)
            btn_false.on_click(on_false)

            display(widgets.VBox([plot_out, widgets.HBox([btn_true, btn_false]), btn_out]))



👉 Station 026017: 1 chunk(s) to review (qc_column=1, qc_flags='238', step=0 days 00:15:00).


In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output